# Bölüm 6 · Parametrik Testler

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b06/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b06/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b06/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b06/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b06/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## Üç çekirdek kazanım ve ileri rota

1. Bağımsız, eşli ve tek örneklem tasarımlarını ayırır.
2. Fark yönü, standart hata ve aralığı birlikte raporlar.
3. Student, Welch ve duyarlılık hedeflerini gerekçelendirir.

**İleri rota:** Yuen, hata kalibrasyonu ve bütün çiftleri dışarıda bırakma ekleri; Bölüm 4–5 temellerinden sonra.

**Ç1 · Çıkış sorusu:** Aynı 18 kişinin önce ve sonra ölçümleri için iki grubu bağımsız saymak uygun mudur?

Yanıtınızı çalışma sonunda öğrenci karar günlüğüne yazın. [Rubrik ve teslim rehberi](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b06/RUBRIK.md).

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b06-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b06-parametrik-testler'
modules=['uygulamalar/b06-parametrik-testler/welch-saglamlik']
expected_main=json.loads((root/"bolumler/b06/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 6 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b06/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 6 — görevler

Güncel kitap bölümünün etkinlik ve alıştırmalarıdır. Önce kendi çözümünüzü
üretin; ardından çözümleri kontrol edin. Kitap sayfa numaraları PDF sürümüne
göre değişebilir.

## D1–D3: Hesaptan Önce Düşünün

1.  Uyku dosyasının 20 satırını 20 bağımsız kişi saymak hangi bilgiyi kaybettirir? Negatif `extra` ne anlama gelir?

2.  Tek örneklem örneğinde sıfır referansı neden anlamlıdır? Sıfır, ayrı bir kontrol grubunun tahmin edilen ortalaması mıdır?

3.  Diş verisinde aynı satır numarası sırasındaki OJ ve VC kayıtlarını neden eşleştiremeyiz? 1 mg/gün seçiminin kapsamını ve bağımsızlık için gereken tasarım bilgisini açıklayın.

Yanıtlar Ek F’de, Bölüm 6 gerçek veri etkinlikleri başlığı altındadır (s. kitaptaki ilgili sayfa).

# Rehberli ve Bağımsız Etkinlikler

## G1–G3: Hesabı Denetleyin

1.  İlaç 1 sütunundan toplamı, ortalamayı, örneklem SS’sini ve %95 aralığı hesaplayın. $p>.05$ sonucunu “ilaç işe yaramıyor” demeden raporlayın.

2.  ID ile 10 farkı bulun. $s_1^2+s_2^2-2s_{12}=s_D^2$ eşitliğini ve $t=d_z\sqrt n$ ilişkisini denetleyin. Saat yerine dakika kullanıldığında fark, aralık, $t$, $p$ ve $d_z$ nasıl değişir?

3.  Doz 1’de Welch serbestlik derecesini yeniden hesaplayın. $n_1=n_2$ iken Welch ve Student t değerlerinin eşit, $p$ değerlerinin farklı olabilmesini formüllerden açıklayın.

## B1–B3: Yeni Dozda Bağımsız Çözüm

Bu kez yalnız 2 mg/gün dozundaki OJ ve VC kayıtlarını seçin; doz 1’in özetlerini taşımayın.

1.  İki grubun $n$, ortalama, SS, OJ$-$VC farkı, Welch $t$, serbestlik derecesi ve %95 aralığını hesaplayın.

2.  Birleştirilmiş SS ile $d$’yi hesaplayın. Büyük $p$ değeri eşdeğerlik kanıtı mıdır? Aralığın izin verdiği değerleri kullanarak açıklayın; önceden belirlenmiş bir önem sınırının eksikliğini tartışın.

3.  En fazla 100 kelimelik bir rapor yazın. Doz 1’de anlamlı, doz 2’de anlamsız fark bulunması neden tek başına doz etkileşimi testinin yerine geçmez?

## H1: Hatalı Raporu Onarın

> “Uyku dosyasında 20 bağımsız kişi vardır. Daha küçük $p$ verdiği için eşli testi seçtik. Levene $p>.05$ olduğundan varyansların eşitliği kanıtlandı. Doz 2’de $p=.964$ olduğu için OJ ve VC kesin eşdeğerdir. %95 aralık hastaların %95’inin farklarını kapsar. ID 9’u silmek $p$’yi küçülttüğünden ana rapora silinmiş analizi koyduk.”

Altı iddiayı tasarım, standart hata, güven aralığı ve raporlama ilkeleriyle düzeltin.

## P1: Yeniden Üretim ve Duyarlılık Projesi

Yerel kaynağın 20/60 satırını ve uyku verisinin 10 ID’sini doğrulayın. Satırları karıştırıp ID ile yeniden eşleştirin; aynı sonuçları elde ettiğinizi gösterin. Her çifti sırayla dışarıda bırakan analizde ortalama fark ve aralıkları raporlayın; tek bir silme sonucunu seçmeyin. Uyku eşleştirme grafiği, fark Q–Q grafiği ve doz 1 OJ/VC dağılımını yorumlayın. Teslim: kaynak/analiz sözlüğü, çalışan kod, grafikler ve bir sayfalık sınırlılık raporu.

<div class="center">

| Ölçüt                                     | Puan |
|:------------------------------------------|-----:|
| Kaynak, analiz birimi ve ID eşleştirmesi  |   20 |
| Üç testin doğru hesaplanması ve fark yönü |   25 |
| Güven aralığı ve etki büyüklüğü paydası   |   20 |
| Tanılar ve bütün çiftler için duyarlılık  |   20 |
| Yeniden çalıştırma ve ölçülü raporlama    |   15 |

</div>

# Alıştırmalar

## Kolay Düzey

1.  Parametrik test genel olarak nedir?

2.  Eşleştirilmiş örneklemler t-testi ne zaman tercih edilir?

3.  Bağımsız örneklemler t-testinde varyans homojenliği neden önemlidir?

4.  Güven aralığının test yorumundaki pratik değeri nedir?

5.  Etki büyüklüğü neden $p$-değerine ek olarak gereklidir?

## Orta Düzey

1.  Bir sınıf ortalaması 70 ölçüt puanıyla karşılaştırılacaktır. Hangi test uygundur?

2.  Farklı yöntemlerle öğrenim gören iki ayrı öğrenci grubu karşılaştırılacaktır. Hangi test kullanılmalıdır?

3.  Aynı katılımcılar uygulama öncesi ve sonrası ölçülmüştür. Hangi test uygundur?

4.  Bağımsız örneklemler t-testinde $t(58)=2.31$, $p=.024$ bulunmuştur. $\alpha=.05$ düzeyinde sonuç ne gösterir?

5.  Levene testi anlamlı çıkmıştır. Bu ne anlama gelir ve araştırmacı neyi dikkate almalıdır?

## İleri Düzey

1.  $n=500$ olan bir çalışmada ortalama farkı anlamlı, Cohen $d=0.08$ bulunmuştur. Nasıl yorumlanır?

2.  Eşleştirilmiş tasarımda ön-test ve son-test ayrı ayrı normal değildir; fark puanları yaklaşık normaldir. Eşleştirilmiş t-testi savunulabilir mi?

3.  İki sınıf bağımsız t-testiyle karşılaştırılmış, fakat öğrenciler sınıflara rastgele atanmamıştır. Hangi sınırlılık raporlanmalıdır?

4.  Tek örneklem t-testinde ortalama farkı için %95 GA $[-1.20,4.80]$ bulunmuştur. Bu aralık karar ile nasıl ilişkilidir?

5.  Bir t-testinden sonra yalnızca “$p<.05$” raporlanmıştır. Daha güçlü rapor için en az dört ek unsur yazınız.


## Revizyon 10: kalibrasyon ve hedef duyarlılığı

## T1–T3: Varsayım, Kalibrasyon ve Hedef Denetimi

1.  $n_1=10,n_2=30$, $\sigma_1=3,\sigma_2=1$ için gerçek ortalama farkı varyansını ve $E(SE_P^2)$’yi hesaplayın. .210145 Student red oranı neden yüksek güç diye yorumlanamaz? Varyansları yer değiştirince yön niçin değişir?

2.  Doz 1’de Welch farkı 5.93, Yuen farkı 6.80’dir. Her kuyruktan yüzde 20 kırpıldığında kişi/grup sayısını, katkı veren sayıyı ve iki yöntemin hedefini ayırın. Daha küçük p’yi ana sonuç seçmek doğru mudur?

3.  Küçük bir örnekte Shapiro p\>.05; başka bir örnekte p\<.05 olsun. Bu iki sonuç neden tek başına Student/Welch/sıra testi seçtirmez? Eşli veride tanının hangi değişkene ait olması gerektiğini açıklayın.


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b06/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b06/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

## Öğrenci kararı ve kısa savunma

**Karar görevi:** Aynı karşılaştırmada Student veya Welch modelini seçin; standart hata, serbestlik derecesi ve varsayım gerekçesini karşılaştırın.

Yalnız bir kararı değiştirin. Ana sonucu koruyun; gerekirse yeni bir hesap hücresi ekleyip denemeyi ayrı kaydedin. Sayısal referansları yeni sonuca uydurmayın. Önce/sonra kanıtını sonuç dosyası, tablo veya açık bir hesapla ilişkilendirin. Yapılmamış hesabı yapılmış gibi yazmayın.

**3–5 dakikalık savunma:** Hangi hedefi seçtiniz ve neden? Tam olarak ne değişti? Hangi belirsizlik ve kapsam sınırı kaldı? Karşılaştırma aynı hipotezi mi sınadı?

Aşağıdaki alanları kendi çalışmanızla doldurun. **Tümünü çalıştır**, bu alanları sizin yerinize tamamlamaz. Sayısal kontrollerin geçmesi öğrenci kararının tamamlandığı anlamına gelmez. Alanların dolu olması da kanıtın doğruluğunu otomatik onaylamaz.

In [ ]:
karar_gunlugu = {
    "bolum": 6,
    "gorev": 'Aynı karşılaştırmada Student veya Welch modelini seçin; standart hata, serbestlik derecesi ve varsayım gerekçesini karşılaştırın.',
    "kararim": "",
    "gerekcem": "",
    "degistirdigim_tek_sey": "",
    "onceki_kanit": "",
    "sonraki_kanit": "",
    "belirsizlik_ve_sinir": "",
    "kisa_savunma": "",
    "cikis_yanitim": ""
}
zorunlu_alanlar = ["kararim","gerekcem","degistirdigim_tek_sey","onceki_kanit","sonraki_kanit","belirsizlik_ve_sinir","kisa_savunma","cikis_yanitim"]
karar_gunlugu["alanlar_dolu"] = all(isinstance(karar_gunlugu[k],str) and bool(karar_gunlugu[k].strip()) for k in zorunlu_alanlar)
karar_gunlugu["durum"] = "Öğretim elemanı incelemesi gerekir" if karar_gunlugu["alanlar_dolu"] else "Öğrenci kararı henüz tamamlanmadı"
print(karar_gunlugu["durum"])
# Bu hücreyi doldurduktan sonra çıktı ZIP'i hücresini yeniden çalıştırın.

In [ ]:
manifest={"chapter":6,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
# OGRETIM_EXPORT_BEGIN
manifest["ogrenci_karari"] = karar_gunlugu
(work/"ogrenci-karari.json").write_text(json.dumps(karar_gunlugu,ensure_ascii=False,indent=2),encoding="utf-8")
# OGRETIM_EXPORT_END
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b06-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"ogrenci-karari.json","ogrenci-karari.json")
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b06-parametrik-testler',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)